In [1]:
# conda activate anndata

import os
import sys
import anndata as ad

sys.path.append("/mnt/lareaulab/reliscu/code")

from junction2psi import *

In [2]:
adata = ad.read_h5ad("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/frontal_cortex/GTEx_frontal_cortex_SJ_counts.h5ad")

In [3]:
SJ_counts_table = pd.DataFrame(adata.X.T, columns=adata.obs_names, index=adata.var_names)
SJ_counts_table.shape

(622617, 269)

In [5]:
expr = pd.read_csv("GTEx_frontal_cortex_counts_TMMF_SampleNetworks/All_10-58-02/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed.csv", index_col=0)
expr.columns = expr.columns.str.replace(".", "-")

In [6]:
SJ_counts_table = SJ_counts_table[expr.columns]
SJ_counts_table.shape

(622617, 200)

In [16]:
events_i1 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I1' in x])
events_i2 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I2' in x])
events_se = pd.Index([x[:-3] for x in SJ_counts_table.index if '_SE' in x])

events = events_i1.intersection(events_i2).intersection(events_se)
i1_events = [x + '_I1' for x in events]
I1_table = SJ_counts_table.loc[i1_events]
I1_table.index = events

i2_events = [x + '_I2' for x in events]
I2_table = SJ_counts_table.loc[i2_events]
I2_table.index = events

se_events = [x + '_SE' for x in events]
SE_table = SJ_counts_table.loc[se_events]
SE_table.index = events
    
I1_filt = I1_table.index[I1_table.sum(axis=1) > 0]
I2_filt = I2_table.index[I2_table.sum(axis=1) > 0]
SE_filt = SE_table.index[SE_table.sum(axis=1) > 0]

filtered_events = I1_filt.intersection(I2_filt).intersection(SE_filt)

I1_table = I1_table.loc[filtered_events]
I2_table = I2_table.loc[filtered_events]
SE_table = SE_table.loc[filtered_events]

psi = ((I1_table + I2_table) /(2*SE_table + I1_table + I2_table)).fillna(0)
reads = SE_table + I1_table + I2_table

In [17]:
psi.shape[0]

28170

In [18]:
psi.head()

,GTEX-13FHO-0011-R10b-SM-5J2MM,GTEX-13JUV-0011-R10b-SM-5LZXR,GTEX-13N1W-0011-R10b-SM-5MR4H,GTEX-16XZZ-0011-R10b-SM-7LT91,GTEX-17JCI-0011-R10b-SM-718A2,GTEX-1H1DG-0011-R10b-SM-CE6S7,GTEX-1JN6P-0011-R10a-SM-F6498,GTEX-1O9I2-0011-R10b-SM-F2I33,GTEX-1PIIG-0011-R10a-SM-F2TB8,GTEX-OXRN-0011-R10A-SM-2I5GC,...,GTEX-1RLM8-0011-R10a-SM-F3414,GTEX-N7MT-0011-R10A-SM-2I3E1,GTEX-NPJ7-0011-R10A-SM-2I3E5,GTEX-QVUS-0011-R10A-SM-3GIK3,GTEX-X261-0011-R10B-SM-4E3JT,GTEX-X4XX-0011-R10B-SM-46MWO,GTEX-ZAB4-0011-R10a-SM-4SOKH,GTEX-ZF28-0011-R10a-SM-4WWEH,GTEX-ZYY3-0011-R10a-SM-GNTAZ,GTEX-ZZPT-0011-R10b-SM-GPI8B
ENSG00000292994_other_2,0.400000,1.0,0.666667,0.666667,0.600000,0.555556,1.0,0.5,0.0,0.00,...,1.0,0.777778,0.538462,0.666667,0.454545,0.5,1.000,0.875000,1.0,1.0
ENSG00000290385_other_1,0.142857,1.0,0.000000,0.000000,0.142857,0.500000,0.0,0.0,1.0,0.04,...,0.0,0.333333,0.111111,0.000000,0.000000,0.0,0.125,0.142857,0.0,0.0
ENSG00000290385_other_2,0.142857,0.0,0.000000,0.000000,0.142857,0.333333,0.2,0.0,0.0,0.04,...,0.2,0.000000,0.111111,0.000000,0.500000,0.0,0.125,0.000000,0.0,0.0
ENSG00000290385_other_3,1.000000,0.0,1.000000,0.833333,0.750000,1.000000,1.0,1.0,0.0,1.00,...,1.0,1.000000,1.000000,0.000000,1.000000,1.0,1.000,1.000000,1.0,1.0
ENSG00000237491_other_16,0.000000,1.0,0.200000,1.000000,1.000000,0.600000,1.0,1.0,1.0,0.00,...,1.0,0.000000,0.666667,1.000000,1.000000,1.0,1.000,1.000000,1.0,1.0


In [ ]:
psi.to_csv(f"data/GTEx_frontal_cortex_SE_PSI.csv")